# 02 · Run the pipeline end-to-end

Take a small unstructured-text corpus and run the **full** workflow to produce an ontology (classes, subclass axioms, object properties), then evaluate it.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # make the package importable
import matplotlib
matplotlib.use('Agg')  # headless-safe; remove for interactive plots
import bottomup_ontology as bo
print('bottomup_ontology', bo.__version__)

bottomup_ontology 0.1.0


## The input corpus (the *unstructured text*)

In [2]:
corpus = [
    'A rugby player plays in a position. Siya Kolisi is a rugby player. '
    'A team has many players.',
    'A club has a coach. The coach trains the team. A player belongs to a club.',
    'Mammals such as lions and impalas live in the savanna. A lion eats impala.',
    'A blog post has a title and comments. A blog post has a category.',
]
gold_classes = {'rugby player','player','team','club','coach','lion',
                'impala','position','blog post','category','mammal'}
len(corpus)

4

## Build the workflow and the shared state, then run it
`run_workflow` topologically sorts the graph and invokes each node's single function on the shared `WorkflowState`.

In [3]:
state = bo.WorkflowState(documents=corpus, gold_classes=gold_classes)
g = bo.build_default_workflow()
state = bo.run_workflow(g, state, verbose=True)

>> running text_cleaning ...
   text_cleaning: produced 62 tokens
>> running preprocessing ...
   preprocessing: scored 17 terms, 26 co-occurrence pairs
>> running term_extraction ...
   term_extraction: 15 candidate classes, 1 composition subclass axioms
>> running relation_extraction ...
   relation_extraction: 10 candidate object properties
>> running axiom_finding ...
   axiom_finding: 3 subclass axioms from Hearst patterns
>> running human_in_the_loop ...
   human_in_the_loop: no approver set — all candidates accepted
>> running evaluation ...
   evaluation: {'ontology': {'classes': 18, 'subclass_axioms': 4, 'object_properties': 10, 'other_axioms': 3}, 'class_prf': {'precision': 0.611, 'recall': 1.0, 'f1': 0.759, 'tp': 11, 'support': 11}, 'data_driven': {'vocab_coverage': 0.824}}


## The resulting ontology

In [4]:
onto = state.ontology
print('summary :', onto.summary())
print()
print('classes :')
for cls in sorted(onto.classes):
    print('   -', cls)

summary : {'classes': 18, 'subclass_axioms': 4, 'object_properties': 10, 'other_axioms': 3}

classes :
   - blog post
   - category
   - club
   - coach
   - comment
   - impala
   - kolisi
   - lion
   - live savanna
   - mammal
   - player
   - position
   - post
   - rugby player
   - savanna
   - siya kolisi
   - team
   - title


In [5]:
print('subclass axioms (X subClassOf Y):')
for sub, sup in sorted(onto.subclass_of):
    print(f'   {sub}  ⊑  {sup}')
print()
print('object properties (domain --predicate--> range):')
for r in onto.object_properties:
    print(f'   {r.subject} --{r.predicate}--> {r.object}  (support={r.support})')
print()
print('axioms found by lexico-syntactic patterns:')
for ax in onto.axioms:
    print('   -', ax)

subclass axioms (X subClassOf Y):
   lion  ⊑  mammal
   live savanna  ⊑  mammal
   rugby player  ⊑  player
   siya kolisi  ⊑  rugby player

object properties (domain --predicate--> range):
   player --play--> position  (support=1)
   kolisi --is--> player  (support=1)
   team --has--> player  (support=1)
   club --has--> coach  (support=1)
   coach --train--> team  (support=1)
   player --belong--> club  (support=1)
   impala --live--> savanna  (support=1)
   lion --eat--> impala  (support=1)
   post --has--> title  (support=1)
   post --has--> category  (support=1)

axioms found by lexico-syntactic patterns:
   - siya kolisi ⊑ rugby player  (lexico-syntactic pattern)
   - lion ⊑ mammal  (lexico-syntactic pattern)
   - live savanna ⊑ mammal  (lexico-syntactic pattern)


## OWL/Turtle serialisation

In [6]:
print(onto.to_turtle())

@prefix : <http://example.org/onto#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:BlogPost a owl:Class ; rdfs:label "blog post" .
:Category a owl:Class ; rdfs:label "category" .
:Club a owl:Class ; rdfs:label "club" .
:Coach a owl:Class ; rdfs:label "coach" .
:Comment a owl:Class ; rdfs:label "comment" .
:Impala a owl:Class ; rdfs:label "impala" .
:Kolisi a owl:Class ; rdfs:label "kolisi" .
:Lion a owl:Class ; rdfs:label "lion" .
:LiveSavanna a owl:Class ; rdfs:label "live savanna" .
:Mammal a owl:Class ; rdfs:label "mammal" .
:Player a owl:Class ; rdfs:label "player" .
:Position a owl:Class ; rdfs:label "position" .
:Post a owl:Class ; rdfs:label "post" .
:RugbyPlayer a owl:Class ; rdfs:label "rugby player" .
:Savanna a owl:Class ; rdfs:label "savanna" .
:SiyaKolisi a owl:Class ; rdfs:label "siya kolisi" .
:Team a owl:Class ; rdfs:label "team" .
:Title a owl:Clas

## Evaluation (Fig. 7.7 *Evaluation* stage)
Gold-standard precision/recall/F1 plus a data-driven vocabulary-coverage assessment.

In [7]:
import json
print(json.dumps(state.evaluation, indent=2))

{
  "ontology": {
    "classes": 18,
    "subclass_axioms": 4,
    "object_properties": 10,
    "other_axioms": 3
  },
  "class_prf": {
    "precision": 0.611,
    "recall": 1.0,
    "f1": 0.759,
    "tp": 11,
    "support": 11
  },
  "data_driven": {
    "vocab_coverage": 0.824
  }
}


## Full execution trace
Every step appends a human-readable line to `state.log`.

In [8]:
for line in state.log:
    print('-', line)

- text_cleaning: produced 62 tokens
- preprocessing: scored 17 terms, 26 co-occurrence pairs
- term_extraction: 15 candidate classes, 1 composition subclass axioms
- relation_extraction: 10 candidate object properties
- axiom_finding: 3 subclass axioms from Hearst patterns
- human_in_the_loop: no approver set — all candidates accepted
- evaluation: {'ontology': {'classes': 18, 'subclass_axioms': 4, 'object_properties': 10, 'other_axioms': 3}, 'class_prf': {'precision': 0.611, 'recall': 1.0, 'f1': 0.759, 'tp': 11, 'support': 11}, 'data_driven': {'vocab_coverage': 0.824}}
